In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle
import warnings, os, json, h5py, tempfile, shutil, traceback

warnings.filterwarnings("ignore")



2026-05-02 21:05:57.655475: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 21:05:57.667148: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755957.680194      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777755957.683847      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 21:05:57.698011: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))

# paths
train_path = "/kaggle/input/lmsys-chatbot-arena/train.csv"
test_path = "/kaggle/input/lmsys-chatbot-arena/test.csv"
sample_sub_path = "/kaggle/input/lmsys-chatbot-arena/sample_submission.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)
df_sample_submission = pd.read_csv(sample_sub_path)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)



TensorFlow Version: 2.18.0
GPU Available: []


2026-05-02 21:06:05.316685: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Train shape: (51729, 9)
Test shape : (5748, 4)


In [3]:
# Concatenate text for tokenization
df_train["text_concat"] = (
    df_train["prompt"]
    + " [SEP] "
    + df_train["response_a"]
    + " [SEP] "
    + df_train["response_b"]
)
df_test["text_concat"] = (
    df_test["prompt"]
    + " [SEP] "
    + df_test["response_a"]
    + " [SEP] "
    + df_test["response_b"]
)

SEQUENCE_LENGTH = 512

# ---------- Tokenizer ----------
tokenizer_path = "/kaggle/input/lmsys-hemkesh-10040-gru/tokenizer_gru.pkl"
if os.path.exists(tokenizer_path):
    try:
        with open(tokenizer_path, "rb") as f:
            text_tokenizer = pickle.load(f)
        print("Loaded external tokenizer.")
    except Exception as e:
        print("Failed to load external tokenizer, will fit a new one:", e)
        text_tokenizer = None
else:
    text_tokenizer = None
    print("External tokenizer not found, will fit a new one.")

if text_tokenizer is None:
    text_tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
    text_tokenizer.fit_on_texts(df_train["text_concat"])
    print("Created new tokenizer with vocab size:", len(text_tokenizer.word_index))

# ---------- Sequences ----------
train_seq = text_tokenizer.texts_to_sequences(df_train["text_concat"])
test_seq = text_tokenizer.texts_to_sequences(df_test["text_concat"])

train_padded = pad_sequences(
    train_seq, maxlen=SEQUENCE_LENGTH, padding="post", truncating="post"
)
test_padded = pad_sequences(
    test_seq, maxlen=SEQUENCE_LENGTH, padding="post", truncating="post"
)

print("Padded train shape:", train_padded.shape)
print("Padded test  shape:", test_padded.shape)



External tokenizer not found, will fit a new one.
Created new tokenizer with vocab size: 263906
Padded train shape: (51729, 512)
Padded test  shape: (5748, 512)


In [4]:
# Target columns
target_cols = ["winner_model_a", "winner_model_b", "winner_tie"]
y_train = df_train[target_cols].values.astype(np.float32)


# ---------- Attempt to load GRU ensemble ----------
def load_model_with_custom_config(model_file_path):
    with h5py.File(model_file_path, "r") as file_handler:
        architecture_config = json.loads(file_handler.attrs["model_config"])
        for layer in architecture_config["config"]["layers"]:
            if layer["class_name"] == "NotEqual":
                for node in layer.get("inbound_nodes", []):
                    if len(node["args"]) > 1 and isinstance(node["args"][1], int):
                        node["args"] = [node["args"][0]]
    with tempfile.NamedTemporaryFile(suffix=".h5", delete=False) as tmp:
        tmp_path = tmp.name
    shutil.copy2(model_file_path, tmp_path)
    with h5py.File(tmp_path, "r+") as f:
        del f.attrs["model_config"]
        f.attrs["model_config"] = json.dumps(architecture_config)

    class NonZeroMask(keras.layers.Layer):
        def __init__(self, convert_to_float=True, **kwargs):
            super().__init__(**kwargs)
            self.convert_to_float = convert_to_float

        def call(self, x):
            if isinstance(x, (list, tuple)):
                x = x[0]
            out = tf.math.not_equal(x, 0)
            return tf.cast(out, tf.float32) if self.convert_to_float else out

    model = keras.models.load_model(
        tmp_path, custom_objects={"NotEqual": NonZeroMask}, compile=False
    )
    os.remove(tmp_path)
    return model


k_fold = 5
ensemble_preds = []
for fold in range(k_fold):
    model_path = f"/kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_{fold}.h5"
    if not os.path.exists(model_path):
        print(f"Fold {fold} model not found at {model_path}")
        continue
    try:
        mdl = load_model_with_custom_config(model_path)
        preds = mdl.predict(test_padded, batch_size=32, verbose=0)
        ensemble_preds.append(preds)
        print(f"Fold {fold} loaded, preds shape {preds.shape}")
    except Exception as e:
        print(f"Failed to load fold {fold}: {e}")
        traceback.print_exc()

# ---------- If no GRU models, train a simple fallback model ----------
if not ensemble_preds:
    print("\nNo GRU models loaded. Training a lightweight fallback model...")
    vocab_size = min(len(text_tokenizer.word_index) + 1, 20000)
    embed_dim = 64
    fallback_model = keras.Sequential(
        [
            keras.layers.Embedding(vocab_size, embed_dim, input_length=SEQUENCE_LENGTH),
            keras.layers.GlobalAveragePooling1D(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(3, activation="softmax"),
        ]
    )
    fallback_model.compile(optimizer="adam", loss="categorical_crossentropy")
    fallback_model.fit(
        train_padded, y_train, epochs=3, batch_size=256, verbose=1, validation_split=0.1
    )
    preds = fallback_model.predict(test_padded, batch_size=32, verbose=0)
    ensemble_preds.append(preds)
    print("Fallback model predictions shape:", preds.shape)

# ---------- Average ensemble ----------
final_ensemble_probabilities = np.mean(np.stack(ensemble_preds, axis=0), axis=0)
print("\nFinal averaged predictions shape:", final_ensemble_probabilities.shape)



Fold 0 model not found at /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_0.h5
Fold 1 model not found at /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_1.h5
Fold 2 model not found at /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_2.h5
Fold 3 model not found at /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_3.h5
Fold 4 model not found at /kaggle/input/lmsys-hemkesh-10040-gru/gru_model_fold_4.h5

No GRU models loaded. Training a lightweight fallback model...
Epoch 1/3
182/182 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: 1.0942 - val_loss: 1.0879
Epoch 2/3
182/182 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0834 - val_loss: 1.0843
Epoch 3/3
182/182 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0608 - val_loss: 1.0993
Fallback model predictions shape: (5748, 3)

Final averaged predictions shape: (5748, 3)


In [5]:
# Ensure probabilities sum to 1 (tiny numerical fixes)
final_ensemble_probabilities = (
    final_ensemble_probabilities
    / final_ensemble_probabilities.sum(axis=1, keepdims=True)
)

# ---------- Create submission ----------
df_submission = df_sample_submission.copy()
df_submission["winner_model_a"] = final_ensemble_probabilities[:, 0]
df_submission["winner_model_b"] = final_ensemble_probabilities[:, 1]
df_submission["winner_tie"] = final_ensemble_probabilities[:, 2]

submission_path = "/kaggle/working/submission.csv"
df_submission.to_csv(submission_path, index=False)

print("\nSubmission saved to:", submission_path)
print("Submission shape:", df_submission.shape)




Submission saved to: /kaggle/working/submission.csv
Submission shape: (5748, 4)


In [6]:
print("\nPrediction statistics:")
for i, col in enumerate(target_cols):
    col_vals = final_ensemble_probabilities[:, i]
    print(
        f"{col} - Mean: {col_vals.mean():.4f}, Std: {col_vals.std():.4f}, "
        f"Min: {col_vals.min():.4f}, Max: {col_vals.max():.4f}"
    )

print("\nRow-wise probability sums (should be 1.0):")
row_sums = final_ensemble_probabilities.sum(axis=1)
print(
    f"Min sum: {row_sums.min():.6f}, Max sum: {row_sums.max():.6f}, Mean sum: {row_sums.mean():.6f}"
)




Prediction statistics:
winner_model_a - Mean: 0.3727, Std: 0.0670, Min: 0.0122, Max: 0.7998
winner_model_b - Mean: 0.3438, Std: 0.0551, Min: 0.0502, Max: 0.6892
winner_tie - Mean: 0.2835, Std: 0.0823, Min: 0.0477, Max: 0.9376

Row-wise probability sums (should be 1.0):
Min sum: 1.000000, Max sum: 1.000000, Mean sum: 1.000000


In [7]:
print("\nInference completed successfully.")
print("Submission file location:", submission_path)


Inference completed successfully.
Submission file location: /kaggle/working/submission.csv
